In [2]:
# TODO: utiliser les prix des avions, les recherches google, les flux événements de l'office du tourisme de Grasse
# TODO: tester les "maj"
# TODO: fixer les chemins relatifs
# TODO: permettre la configuration du PACE aisément
# TODO:
# Concepte de booking curve (selon l'historique des réservations)
# Scrape les disponibilités des concurrents et stocker
# Scrape les prix des concurrents et stocker"""


# Taux de remplissage: Faible, Moyen, Fort, Critique

# Montée en charge
# Pace (Matrice des seuils)

"""Phase 1 : Le Socle (J-90 et plus)
​Objectif : Assurer une base de sécurité (couvrir les coûts fixes).
​Si "Faible" : On ne panique pas, c'est trop tôt. On maintient les prix publics, on ouvre les promotions "Early Bird" (Réservez tôt, moins cher, non remboursable).
​Si "Fort" (> 40%) : Attention ! Vous vendez trop vite et trop peu cher. Action : Augmentez le prix de base (+15%) et fermez les tarifs "Corporate" ou "Groop" bon marché.
​Phase 2 : Le Yielding Actif (J-21 à J-60)
​Objectif : Optimiser le prix moyen. C'est là que le vrai jeu commence.
​Si "Faible" (Retard) : Il faut stimuler. Lancez une "Flash Sale" ou une offre packagée (Chambre + Petit-déj inclus) pour augmenter la valeur perçue sans casser le prix sec.
​Si "Fort" (Avance) : Vous êtes en position de force. Alignez-vous sur le concurrent le plus cher de votre set. Refusez les séjours d'une seule nuit (Minimum Stay 2 nuits) pour optimiser le planning.
​Phase 3 : L'Optimisation Finale (J-0 à J-14)
​Objectif : Remplir les trous ou maximiser la marge pure.
​Si "Faible" (< 60%) : C'est l'alerte rouge. C'est le moment d'ouvrir les vannes sur les OTAs (Booking/Expedia) avec des promos de dernière minute ("Last Minute Deal"). Mieux vaut vendre à -20% que de laisser la chambre vide (tant qu'on est au-dessus du coût variable).
​Si "Critique" (> 90%) : Fermez les canaux de vente coûteux (Booking.com prend 17% de commission). Gardez les dernières chambres uniquement pour votre site web en direct et vendez-les au prix fort ("Rack Rate").

​1. Levier Tarifaire (Le Prix)
​C'est le levier le plus évident, mais attention à ne pas détruire votre image de marque.
​Pour accélérer (Retard) :
​Promotions Opaque : Activez les promos "Mobiles" ou "Genius" sur Booking.com. Cela baisse le prix pour des segments spécifiques sans afficher un prix barré public à tout le monde.
​Offres Packagées : "3 nuits pour le prix de 2" ou "Petit-déjeuner offert". Vous maintenez votre ADR (Prix moyen) facial, mais vous baissez le coût réel pour le client.
​Pour freiner (Avance) :
​Monter le BAR (Best Available Rate) : Augmentez le prix de 10€ à 20€.
​Fermer les tarifs réduits : Bloquez les tarifs "Non Remboursable" (souvent -10%) pour ne laisser que le tarif Flexible (plus cher).
​2. Levier de Restrictions (Inventory Control)
​C'est souvent plus puissant que le prix.
​Pour accélérer (Retard) :
​Lever le MLOS (Minimum Length of Stay) : Si vous imposiez 2 nuits minimum, passez à 1 nuit.
​Accepter le "Samedis isolés" : Souvent, les hôteliers bloquent les arrivées le samedi pour éviter les séjours d'une nuit qui bloquent le week-end. Si vous êtes en retard : ouvrez tout.
​Pour freiner (Avance) :
​Imposer un MLOS : "2 nuits minimum". Cela filtre les clients "parasites" qui prennent juste le samedi soir (et empêchent de vendre le vendredi-samedi).
​CTA (Closed to Arrival) : Interdire d'arriver le jour J, forçant les gens à arriver la veille.
​3. Levier de Distribution (Les Canaux)
​Gérez où vous vendez.
​Pour accélérer (Retard) :
​Ouvrir tous les canaux : Expedia, Booking, HotelTonight (pour le last minute).
​Surcommissionner (Visibility Booster) : Payer 20% de commission au lieu de 17% sur Booking.com pour remonter en haut de page temporairement.
​Pour freiner (Avance) :
​Fermer les OTAs coûteux : Si vous êtes presque complet, fermez Expedia et Booking. Ne gardez que votre Site Web (0% commission) et le téléphone. C'est là que vous faites votre marge nette.

​Une fois que vous avez cette colonne Pickup, vous pouvez affiner votre stratégie :
​Pickup Nul (0) : Calme plat.
​Action : Si ça dure depuis 3 jours \rightarrow Baisse de prix.
​Pickup Positif fort (> 3/jour) : Demande forte soudaine.
​Action : Monter le prix immédiatement pour les chambres restantes.
​Pickup Négatif (Annulations) : Attention.
​Action : Vérifiez si un concurrent n'a pas cassé ses prix, incitant vos clients à annuler pour aller ailleurs.
"""

''

In [1]:
from price import *

In [ ]:
def calcul_price_df(df):
    now = pd.Timestamp.now()
    df["ecart_TO"] = df.apply(lambda row: calcul_ecart((row.name-now).days, row["Libre"], row["Dispo"]), axis=1)
    print(f"ecart_TO: {df["ecart_TO"]}")
    mask_maj_vide = df["maj"].isna()
    mask_maj_is_number = pd.to_numeric(df["maj"], errors="coerce").notna()
    mask_maj_is_list = (df["maj"].str.startswith("[", na=False)) & (df["maj"].str.endswith("]", na=False))

    print(f"Avant calcul: {df.loc[["2025-12-04"]]}")
    # Calcul du prix pour les lignes sans "maj"
    df.loc[mask_maj_vide, "Aroma"] = df[mask_maj_vide].apply(
        lambda row: calcul_price(
            row["ecart_TO"], 
            row[list(SHORT_NAMES)].tolist()
        ), axis=1
    )
    
    print(f"après calcul maj vide: {df.loc[["2025-12-04"]]}")
    # Calcul du prix pour les lignes avec "maj" nombre: 10 -> +10% -> prix * 1.10
    df.loc[mask_maj_is_number, "Aroma"] = df[mask_maj_is_number].apply(
        lambda row: (1 + pd.to_numeric(row["maj"]) / 100) * calcul_price(
            row["ecart_TO"], 
            row[list(SHORT_NAMES)].tolist()
        ), axis=1
    )

    print(f"après calcul maj is number: {df.loc[["2025-12-04"]]}")
    # Calcul du prix pour les lignes avec "maj" liste: [75, 100, 150] -> ignore le prix des concurrents, utilise la liste à la place.
    df.loc[mask_maj_is_list, "Aroma"] = df[mask_maj_is_list].apply(
        lambda row: calcul_price(
            row["ecart_TO"], 
            json.loads(row["maj"])
        ), axis=1
    )
    print(f"après calcul maj is liste: {df.loc[["2025-12-04"]]}")

In [17]:
df = load_df()
calcul_price_df(df.loc[["2025-12-04"]])


ecart_TO: 2025-12-04    11.0
Name: ecart_TO, dtype: float64
Avant calcul:            Evenement  maj  Dispo  Libre  Aroma  ecart_TO  BnBMouans  LaPoste  \
2025-12-04       NaN  NaN   24.0    7.0   45.0      11.0        NaN     72.0   

            IbisCannes  Casabella  Bellaudiere  IbisMouans  BestWestern  
2025-12-04         NaN        NaN          NaN         NaN         75.0  
après calcul maj vide:            Evenement  maj  Dispo  Libre  Aroma  ecart_TO  BnBMouans  LaPoste  \
2025-12-04       NaN  NaN   24.0    7.0   45.0      11.0        NaN     72.0   

            IbisCannes  Casabella  Bellaudiere  IbisMouans  BestWestern  
2025-12-04         NaN        NaN          NaN         NaN         75.0  
après calcul maj is number:            Evenement  maj  Dispo  Libre  Aroma  ecart_TO  BnBMouans  LaPoste  \
2025-12-04       NaN  NaN   24.0    7.0   45.0      11.0        NaN     72.0   

            IbisCannes  Casabella  Bellaudiere  IbisMouans  BestWestern  
2025-12-04         NaN

In [12]:
type(df.loc["2025-12-04", :])

pandas.core.series.Series

In [18]:
from price import *

In [21]:
pd.Timestamp.now().normalize()

Timestamp('2025-11-23 00:00:00')

In [22]:
df = load_df()
df

,Evenement,maj,Dispo,Libre,Aroma,ecart_TO,BnBMouans,LaPoste,IbisCannes,Casabella,Bellaudiere,IbisMouans,BestWestern
2025-11-20,NaN,NaN,24,11,56.00,-41,56.0,60.0,67.0,70.0,88.0,94.0,96.0
2025-11-21,NaN,NaN,24,9,56.00,-32,56.0,88.0,67.0,67.0,84.0,85.0,84.0
2025-11-22,NaN,NaN,24,3,71.95,-8,56.0,76.0,67.0,60.0,88.0,85.0,84.0
2025-11-23,NaN,-15,24,14,47.60,-53,56.0,67.0,67.0,64.0,84.0,85.0,84.0
2025-11-24,NaN,NaN,24,10,51.00,-33,51.0,72.0,60.0,64.0,128.0,105.0,96.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-11-18,NaN,NaN,32,31,96.70,-7,NaN,NaN,NaN,103.0,NaN,106.0,NaN
2026-11-19,NaN,NaN,32,31,95.30,-7,51.0,NaN,NaN,103.0,NaN,101.0,NaN
2026-11-20,NaN,NaN,32,31,87.60,-7,NaN,NaN,NaN,103.0,113.0,90.0,NaN
2026-11-21,NaN,NaN,32,31,87.60,-7,52.0,NaN,NaN,103.0,113.0,90.0,NaN


In [23]:
now = pd.Timestamp.now().normalize()